# L5: Self-Reflecting Agents with Loops

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
from dotenv import load_dotenv

_ = load_dotenv()

In [4]:
from typing import List
from colorama import Fore
from haystack import Pipeline, component
from haystack.components.builders.prompt_builder import PromptBuilder
from haystack.components.generators.openai import OpenAIGenerator

### Create an EntitiesValidator

In [5]:
@component
class EntitiesValidator:
    
    @component.output_types(entities_to_validate=str, entities=str)
    def run(self, replies: List[str]):
        if 'DONE' in replies[0]:
            return {"entities": replies[0].replace('DONE', '')}
        else:
            print(Fore.RED + "Reflecting on entities\n", replies[0])
            return {"entities_to_validate": replies[0]}

In [6]:
entities_validator = EntitiesValidator()
entities_validator.run(replies=["{'name': 'Tuana'}"])

Reflecting on entities
 {'name': 'Tuana'}


{'entities_to_validate': "{'name': 'Tuana'}"}

In [7]:
entities_validator.run(replies=["DONE {'name': 'Tuana'}"])

{'entities': " {'name': 'Tuana'}"}

### Create a Prompt Template with an 'if' block

In [8]:
template = """
{% if entities_to_validate %}
    Here was the text you were provided:
    {{ text }}
    Here are the entities you previously extracted:
    {{ entities_to_validate[0] }}
    Are these the correct entities?
    Things to check for:
    - Entity categories should exactly be "Person", "Location" and "Date"
    - There should be no extra categories
    - There should be no duplicate entities
    - If there are no appropriate entities for a category, the category should have an empty list
    If you are done say 'DONE' and return your new entities in the next line
    If not, simply return the best entities you can come up with.
    Entities:
{% else %}
    Extract entities from the following text
    Text: {{ text }}
    The entities should be presented as key-value pairs in a JSON object.
    Example:
    {
        "Person": ["value1", "value2"],
        "Location": ["value3", "value4"],
        "Date": ["value5", "value6"]
    }
    If there are no possibilities for a particular category, return an empty list for
    this category
    Entities:
{% endif %}
"""

### Create A Self-Reflecting Agent

In [ ]:
prompt_template = PromptBuilder(template=template)
llm = OpenAIGenerator()
entities_validator = EntitiesValidator()

self_reflecting_agent = Pipeline(max_runs_per_component=10)

self_reflecting_agent.add_component("prompt_builder", prompt_template)
self_reflecting_agent.add_component("entities_validator", entities_validator)
self_reflecting_agent.add_component("llm", llm)

self_reflecting_agent.connect("prompt_builder.prompt", "llm.prompt")
self_reflecting_agent.connect("llm.replies", "entities_validator.replies")
self_reflecting_agent.connect("entities_validator.entities_to_validate", "prompt_builder.entities_to_validate")

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


TypeError: PipelineBase.__init__() got an unexpected keyword argument 'max_loops_allowed'